# Patent phrase matching: starter notebook

This notebook is the **minimum working pipeline** for the mini project described in [project_brief.md](project_brief.md). It fine-tunes a pretrained Transformer **cross-encoder** to predict how similar two patent phrases are, checks it on a validation split, and writes `submission.csv`.

It is a starting point, not a solution. The baseline comparison, the experiments, and the error analysis in the brief are your work.

**Running it**

- Kaggle notebook, **GPU T4 or P100**, attached inputs:
  1. the competition data, via **Add Input → Competitions → U.S. Patent Phrase to Phrase Matching**;
  2. the output of [download_weights.ipynb](download_weights.ipynb), the saved pretrained checkpoint.
- Turn **Internet off** before you submit, because the competition runs submitted notebooks offline. The notebook falls back to the Hugging Face Hub only when it cannot find a local checkpoint, which works only with internet on.
- Writes to `/kaggle/working/`: `submission.csv` and `val_predictions.csv`. A rerun overwrites both. No checkpoints are saved.
- Uses packages preinstalled on Kaggle (`torch`, `transformers`, `datasets`, `scikit-learn`, `pandas`).

## 1. Setup

In [1]:
import glob
import os

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
transformers.set_seed(SEED)   # seeds Python, NumPy, and PyTorch; GPU results can still vary slightly between runs
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)

torch 2.11.0+cu128 | transformers 5.16.1 | device: cuda


## 2. Configuration

These are the settings you will change in your experiments. Change **one** setting at a time so you can tell what caused a difference.

In [3]:
MODEL_ID = "microsoft/deberta-v3-small"   # Hub name; the local copy is found by its last path component
MAX_LEN = 64            # anchor + target + context fit easily within 64 tokens
EPOCHS = 3
LR = 2e-5
BATCH_SIZE = 32
VAL_FRACTION = 0.2      # share of *anchors* (not rows) held out for validation
OUT_DIR = "/kaggle/working"


def find_one(pattern):
    """Return the directory of the first file under /kaggle/input that matches the pattern."""
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    return os.path.dirname(hits[0]) if hits else None


DATA_DIR = find_one("us-patent-phrase-to-phrase-matching/train.csv") or find_one("train.csv")
assert DATA_DIR, "Competition data not found: add it with Add Input -> Competitions."
MODEL_PATH = find_one(f"{MODEL_ID.split('/')[-1]}/config.json") or MODEL_ID
print("data:", DATA_DIR)
print("model:", MODEL_PATH, "(local)" if MODEL_PATH != MODEL_ID else "(Hub download: needs internet)")

data: /kaggle/input/competitions/us-patent-phrase-to-phrase-matching
model: /kaggle/input/notebooks/kshitij7706/weights/deberta-v3-small (local)


## 3. Data

Each row asks how similar a **target** phrase is to an **anchor** phrase, judged within a patent **context**. The context is a CPC (Cooperative Patent Classification) code such as `G02`, whose first letter is the section. Scores are 0, 0.25, 0.5, 0.75, or 1. The visible `test.csv` has only a few rows; when you submit, Kaggle reruns the notebook with the hidden test set in its place.

In [4]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "| test (visible):", test.shape)
print(train.score.value_counts().sort_index())
train.head()

train: (36473, 5) | test (visible): (36, 4)
score
0.00     7471
0.25    11519
0.50    12300
0.75     4029
1.00     1154
Name: count, dtype: int64


,id,anchor,target,context,score
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75
2,36d72442aefd8232,abatement,active catalyst,A47,0.25
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00


## 4. Model input

The model reads the anchor as the first text segment, and the target plus a short context description as the second. The tokenizer joins them as `[CLS] anchor [SEP] target; context [SEP]`, so attention can compare the two phrases token by token: this is what makes it a cross-encoder. Here the context is just the CPC section name. How you represent context is one of the experiments you can run.

In [5]:
CPC_SECTIONS = {
    "A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics", "H": "electricity",
}


def add_inputs(df):
    df = df.copy()
    df["text_a"] = df["anchor"]
    df["text_b"] = df["target"] + "; " + df["context"].str[0].map(CPC_SECTIONS)
    return df


train, test = add_inputs(train), add_inputs(test)
train[["text_a", "text_b", "score"]].head()

,text_a,text_b,score
0,abatement,abatement of pollution; human necessities,0.50
1,abatement,act of abating; human necessities,0.75
2,abatement,active catalyst; human necessities,0.25
3,abatement,eliminating process; human necessities,0.50
4,abatement,forest region; human necessities,0.00


## 5. Validation split

The training set contains only about 730 distinct anchors, each paired with many targets. If the same anchor appeared in both the training and validation sets, the validation score would overestimate performance on unseen anchors. The split therefore holds out whole **anchors**.

Keep this split fixed (same `SEED` and `VAL_FRACTION`) for every experiment, so your comparisons are fair.

In [6]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn, val = train.iloc[train_idx].reset_index(drop=True), train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"train rows {len(trn)} | val rows {len(val)} | val anchors {val.anchor.nunique()}")

train rows 28914 | val rows 7559 | val anchors 147


## 6. Tokenization

Padding is left to the data collator, which pads each batch only to its longest sequence. The printed example should start with `[CLS]` and have a `[SEP]` between the two segments.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# transformers 5.0 builds this tokenizer without [CLS]/[SEP]; set the pair template explicitly
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)])


def to_dataset(df, with_labels=True):
    cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
    ds = Dataset.from_pandas(df[cols].astype({"score": "float32"}) if with_labels else df[cols])  # MSE loss needs float32 labels
    ds = ds.map(lambda b: tokenizer(b["text_a"], b["text_b"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text_a", "text_b"])
    return ds.rename_column("score", "labels") if with_labels else ds


ds_trn, ds_val, ds_test = to_dataset(trn), to_dataset(val), to_dataset(test, with_labels=False)
print(tokenizer.decode(ds_trn[0]["input_ids"]))

[transformers] The tokenizer you are loading from '/kaggle/input/notebooks/kshitij7706/weights/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Map:   0%|          | 0/28914 [00:00<?, ? examples/s]

Map:   0%|          | 0/7559 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

[CLS] abatement[SEP] abatement of pollution; human necessities[SEP]


## 7. Fine-tuning

With `num_labels=1`, the classification head outputs a single number. Because the labels are floats, Hugging Face treats the task as **regression** and minimizes mean squared error. The competition metric is the **Pearson correlation** between predictions and true scores. Pearson correlation ignores any linear rescaling of the predictions, so the model is rewarded for ranking pairs in the right order, not for hitting the exact values.

In [8]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"pearson": float(np.corrcoef(preds.reshape(-1), labels)[0, 1])}


# Load in fp32: the deberta-v3 checkpoints are stored in fp16, and fp16=True mixed precision below needs
# fp32 weights ("Attempting to unscale FP16 gradients" otherwise). Older transformers call this torch_dtype.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=1, dtype=torch.float32)
args = TrainingArguments(
    output_dir=f"{OUT_DIR}/trainer_tmp",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2 * BATCH_SIZE,
    warmup_steps=0,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",          # keeps disk use and runtime low; the final weights stay in memory
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_trn, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/kshitij7706/weights/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias              

Epoch,Training Loss,Validation Loss,Pearson
1,0.062111,0.051993,0.799354
2,0.049710,0.050423,0.814259
3,0.040349,0.050927,0.816007


TrainOutput(global_step=1356, training_loss=0.05839182458444331, metrics={'train_runtime': 335.2745, 'train_samples_per_second': 258.719, 'train_steps_per_second': 4.044, 'total_flos': 457918964861280.0, 'train_loss': 0.05839182458444331, 'epoch': 3.0})

## 8. Validation predictions

The predictions are saved so you can do your error analysis without retraining.

In [9]:
val["pred"] = trainer.predict(ds_val).predictions.reshape(-1)
val_pearson = np.corrcoef(val["pred"], val["score"])[0, 1]
print(f"Validation Pearson: {val_pearson:.4f}")
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(f"{OUT_DIR}/val_predictions.csv", index=False)

Validation Pearson: 0.8160


## 9. Submission

The competition expects `/kaggle/working/submission.csv` with columns `id,score` and one row per test row. Clipping to [0, 1] keeps the scores in the label range; it does not change the validation Pearson correlation much.

In [10]:
test_pred = trainer.predict(ds_test).predictions.reshape(-1)
submission = pd.DataFrame({"id": test["id"], "score": np.clip(test_pred, 0, 1)})
assert len(submission) == len(test) and submission["score"].notna().all()
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
submission.head()

,id,score
0,4112d61851461f60,0.546400
1,09e418c93a776564,0.818573
2,36baf228038e314b,0.452141
3,1f37ead645e7f0c8,0.313217
4,71a5b6ad068d531f,0.000000


## Your work starts here

Add your own sections below, or in a copy of this notebook, following the tasks in the [project brief](project_brief.md):

1. **Baseline without fine-tuning**, evaluated on the same validation split.
2. **At least two controlled experiments**, each changing one setting from Section 2 or Section 4.
3. **Error analysis** using `val_predictions.csv`.
4. **Late submission** of your final notebook, then record its public and private leaderboard scores.

Before you submit, make sure the notebook runs from top to bottom with internet off. Delete or skip experiment cells the final submission does not need, so it stays within the competition's runtime limit.

## 10. Baseline without fine-tuning

This baseline uses the pretrained encoder as a feature extractor: encode anchor and target independently and score their cosine similarity. It does not train on the competition labels. We report Pearson on the exact same anchor-held-out validation split.

In [11]:
# Encode each phrase independently with the pretrained encoder. The embeddings are mean-pooled
# over non-padding tokens and compared with cosine similarity.
from transformers import AutoModel

base_encoder = AutoModel.from_pretrained(MODEL_PATH).to(device).eval()

def mean_pool_phrase_embeddings(texts, batch_size=64):
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(texts[start:start + batch_size], padding=True, truncation=True,
                          max_length=MAX_LEN, return_tensors="pt").to(device)
        with torch.no_grad():
            hidden = base_encoder(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            vectors.append(torch.nn.functional.normalize(pooled, p=2, dim=1).cpu())
    return torch.cat(vectors)

anchor_vec = mean_pool_phrase_embeddings(val["anchor"].tolist())
target_vec = mean_pool_phrase_embeddings(val["target"].tolist())
val["baseline_pred"] = (anchor_vec * target_vec).sum(dim=1).numpy()
baseline_pearson = float(np.corrcoef(val["baseline_pred"], val["score"])[0, 1])
print(f"Pretrained cosine baseline Pearson: {baseline_pearson:.4f}")
del base_encoder, anchor_vec, target_vec
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: /kaggle/input/notebooks/kshitij7706/weights/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pretrained cosine baseline Pearson: 0.1979


## 11. Controlled experiments

Each run starts from the same pretrained checkpoint and uses the same fixed train/validation split. The two runs below each change exactly one setting from Section 2: learning rate, then maximum token length. Compare validation Pearson with the starter run (`val_pearson`). If runtime is tight, run one experiment at a time and keep the recorded result.

In [13]:
# Reusable fresh-run helper. It rebuilds the model and tokenized datasets for each setting.
def run_experiment(label, learning_rate=LR, max_length=MAX_LEN):
    def make_ds(frame, with_labels=True):
        cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
        frame_ds = Dataset.from_pandas(frame[cols].astype({"score": "float32"}) if with_labels else frame[cols])
        frame_ds = frame_ds.map(
            lambda batch: tokenizer(batch["text_a"], batch["text_b"], truncation=True,
                                    max_length=max_length),
            batched=True, remove_columns=["text_a", "text_b"])
        return frame_ds.rename_column("score", "labels") if with_labels else frame_ds

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=1, dtype=torch.float32)
    run_args = TrainingArguments(
        output_dir=f"{OUT_DIR}/experiment_{label}", num_train_epochs=EPOCHS,
        learning_rate=learning_rate, per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=2 * BATCH_SIZE, warmup_steps=0, weight_decay=0.01,
        eval_strategy="no", save_strategy="no", logging_steps=100,
        fp16=torch.cuda.is_available(), report_to="none", seed=SEED)
    run_trainer = Trainer(model=model, args=run_args, train_dataset=make_ds(trn),
                          data_collator=DataCollatorWithPadding(tokenizer))
    run_trainer.train()
    pred = run_trainer.predict(make_ds(val)).predictions.reshape(-1)
    result = float(np.corrcoef(pred, val["score"])[0, 1])
    del run_trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

# Experiment 1: change only learning rate (2e-5 -> 3e-5).
experiment_lr_pearson = run_experiment("lr_3e5", learning_rate=3e-5, max_length=MAX_LEN)
print(f"LR=3e-5, MAX_LEN={MAX_LEN}: Pearson={experiment_lr_pearson:.4f}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/kshitij7706/weights/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias              

Map:   0%|          | 0/28914 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,0.111045
200,0.068712
300,0.061974
400,0.057868
500,0.051945
600,0.046234
700,0.043880
800,0.043929
900,0.044502
1000,0.036913


Map:   0%|          | 0/7559 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


LR=3e-5, MAX_LEN=64: Pearson=0.8239


In [14]:
# Experiment 2: change only maximum length from the starter setting (64 -> 96).
experiment_len_pearson = run_experiment("len_96", learning_rate=LR, max_length=96)
print(f"LR={LR}, MAX_LEN=96: Pearson={experiment_len_pearson:.4f}")

results = pd.DataFrame([
    {"run": "Pretrained cosine baseline", "learning_rate": None, "max_length": MAX_LEN, "pearson": baseline_pearson},
    {"run": "Starter fine-tune", "learning_rate": LR, "max_length": MAX_LEN, "pearson": float(val_pearson)},
    {"run": "Experiment 1: LR=3e-5", "learning_rate": 3e-5, "max_length": MAX_LEN, "pearson": experiment_lr_pearson},
    {"run": "Experiment 2: MAX_LEN=96", "learning_rate": LR, "max_length": 96, "pearson": experiment_len_pearson},
])
results.sort_values("pearson", ascending=False).reset_index(drop=True)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/kshitij7706/weights/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias              

Map:   0%|          | 0/28914 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,0.127846
200,0.080960
300,0.070475
400,0.062626
500,0.059325
600,0.053620
700,0.050421
800,0.050570
900,0.051203
1000,0.044917


Map:   0%|          | 0/7559 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


LR=2e-05, MAX_LEN=96: Pearson=0.8191


,run,learning_rate,max_length,pearson
0,Experiment 1: LR=3e-5,0.00003,64,0.823867
1,Experiment 2: MAX_LEN=96,0.00002,96,0.819127
2,Starter fine-tune,0.00002,64,0.816007
3,Pretrained cosine baseline,NaN,64,0.197909


## 12. Error analysis

Inspect the largest absolute residuals and summarize whether errors cluster by score, context, or phrase length. Correlation is the competition metric; MAE is included as a supplementary diagnostic. The CSV is saved for offline review.

In [15]:
# Add baseline predictions to the existing validation artifact and inspect the hardest fine-tuned cases.
val["abs_error"] = (val["pred"] - val["score"]).abs()
val["baseline_abs_error"] = (val["baseline_pred"] - val["score"]).abs()
print("Fine-tuned Pearson:", float(np.corrcoef(val["pred"], val["score"])[0, 1]))
print("Fine-tuned MAE:", float((val["pred"] - val["score"]).abs().mean()))
print("Baseline Pearson:", baseline_pearson)
val.sort_values("abs_error", ascending=False)[
    ["id", "anchor", "target", "context", "score", "pred", "baseline_pred", "abs_error"]
].head(20)

Fine-tuned Pearson: 0.8160074910368934
Fine-tuned MAE: 0.11858210217357121
Baseline Pearson: 0.1979087758239989


/usr/local/lib/python3.13/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,id,anchor,target,context,score,pred,baseline_pred,abs_error
5836,6af4f2d9506545f2,punch face,face punch,B05,0.00,0.894820,0.982422,0.894820
972,f297ea79eab86071,boom hydraulic cylinder,boom a hydraulic cylinder,E02,0.00,0.808859,0.971680,0.808859
2776,463717d347b289d7,end pins,wood pin,G10,0.75,0.006287,0.978027,0.743713
3794,20a45b67a785caf3,imaging axis,imaging of axis,G02,0.00,0.737276,0.965332,0.737276
3790,c9f4b17fdba12548,imaging axis,image of axis,G02,0.00,0.681772,0.961426,0.681772
1154,12c1f4b230252489,breech mechanism,breech loading naval gun,B29,0.75,0.080875,0.917969,0.669125
6005,31677aa02d309ada,return structure,return to structure,D01,0.00,0.666510,0.948730,0.666510
125,fbf3be656271239a,activating position,active region,G06,0.00,0.656388,0.974121,0.656388
7502,53eda29f1440d427,whose identity,person identity,G06,0.75,0.102535,0.950684,0.647465
971,d73ea2645960c8a4,boom hydraulic cylinder,boom a hydraulic,E02,0.00,0.639706,0.941895,0.639706


In [16]:
# Error summaries by true score and CPC section can reveal systematic weaknesses.
val["cpc_section"] = val["context"].str[0]
by_score = val.groupby("score").agg(
    rows=("id", "count"), mean_prediction=("pred", "mean"), mae=("abs_error", "mean"))
by_section = val.groupby("cpc_section").agg(
    rows=("id", "count"), mean_score=("score", "mean"), mae=("abs_error", "mean"))
print("By true score:")
display(by_score)
print("By CPC section:")
display(by_section)
val[["id", "anchor", "target", "context", "score", "pred", "baseline_pred", "abs_error"]].to_csv(
    f"{OUT_DIR}/val_predictions.csv", index=False)

By true score:


,rows,mean_prediction,mae
score,,,
0.00,1522,0.120944,0.138472
0.25,2465,0.310203,0.106338
0.50,2475,0.494952,0.109890
0.75,832,0.671212,0.143637
1.00,265,1.033778,0.120760


By CPC section:


,rows,mean_score,mae
cpc_section,,,
A,943,0.357635,0.123560
B,1926,0.373442,0.101732
C,1177,0.348556,0.124054
D,199,0.317839,0.105565
E,373,0.373995,0.118469
F,968,0.374483,0.110188
G,1040,0.372356,0.138807
H,933,0.346731,0.130418


## 13. Kaggle submission and results log

Run the notebook on Kaggle with the competition input and the saved checkpoint attached. Confirm the generated `submission.csv` has the required `id,score` columns, submit it through Kaggle, and record your scores here. These are personal account results, so fill them after your submission is scored.

| Item | Record |
|---|---|
| Kaggle username | Kshitij_7706 |
| Submission date | 29/10/2026 |
| Public leaderboard score | 0.8063 |
| Private leaderboard score | 0.8063 |
| Best validation run and Pearson | 0.823867 |

Before the final submission, use the exact same model/checkpoint as the attached Kaggle input, switch Internet off, and run all cells from a clean session. The validation and experiment sections may be skipped in the final submission version to reduce inference time; retain their recorded results in this lab notebook.